# QuantiPhy · IIIXR 2D Motion (`g_box`) · Full 18 câu / 6 video trên GPU T4 ×2

Chạy toàn bộ 18 câu hỏi chuyển động 2D chuẩn (`142, 143, 398–401, 427–434, 1896, 2153, 2233, 2234`) trên 6 video QuantiPhy validation:
- **Stage 1 (`Qwen3-VL-8B-Instruct`, shard qua 2×T4 FP16):** Parse fallback + grounding `--notext` cho nhánh `v3nt`, giải phóng toàn bộ VRAM khi xong.
- **Stage 2 (2 worker song song, 1 T4/worker, chia theo video):** GroundingDINO + OWLv2 + SAM 2.1 + Scene Ruler, dùng chung track cấp video cho các câu cùng video nhưng tách riêng parse, vật chuẩn và phép tính từng câu.
- **Stage 3 & Evaluator độc lập:** Đo động học `measure_v3` + `xground` trên CPU, khóa prediction rồi mới ghép nhãn ground truth, `cached_box` (18 file `mot_val_*.json`) và `caw` (đọc trực tiếp `run/predictions.csv` từ `quantiphy_full.tar` nếu gắn vào `/kaggle/input`).

Kết quả xuất `/kaggle/working/iiixr_box_full.tar`, `iiixr_summary.json`, `per_question.csv` (đủ 18 dòng) và trang phân tích lỗi offline `run/index.html`.

## 1. Cấu hình

In [ ]:
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
SOURCE_REF = "main"  # Có thể pin trực tiếp commit SHA cụ thể
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # Chỉ dùng nếu repo chuyển sang Private

# Để trống để tự tải snapshot đã pin từ HuggingFace; hoặc điền đường dẫn trong /kaggle/input khi chạy offline.
DATASET_DIR = ""
MODELS_DIR = ""
CAW_FULL_TAR = ""  # Để trống để tự tìm quantiphy_full.tar trong /kaggle/input nếu đã gắn
WHEELHOUSE = ""

SCOPE = "target18"         # "target18" (18 câu chuẩn) hoặc "survey158" (đánh dấu unsupported ngoài phạm vi)
DTYPE_PROFILE = "t4_fp16"  # "t4_fp16", "fp32", hoặc "upstream_bf16"
RULE_PARSE_ONLY = False
PARTITION_COUNT = 1        # Tăng >1 nếu muốn chia phiên theo video
PARTITION_INDEX = 0
OUTPUT_BUDGET_GIB = 18.0

## 2. Clone repo tại ref đã chọn & cài đặt dependencies đã khóa

In [ ]:
import base64, os, subprocess, sys, tempfile
from pathlib import Path

source = Path(tempfile.mkdtemp(prefix="iiixr-box-")) / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
clone = ["git", "-c", "credential.helper=", "clone", "--quiet", REPO_URL, str(source)]
if subprocess.run(clone, env=env).returncode:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    subprocess.run(clone, check=True, env=dict(env, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader",
        GIT_CONFIG_VALUE_0="Authorization: Basic " + auth))
    del token, auth

remote_ref = f"refs/remotes/origin/{SOURCE_REF}"
is_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", remote_ref], cwd=source).returncode == 0
subprocess.run(["git", "checkout", "--quiet", "--detach", remote_ref if is_branch else SOURCE_REF], cwd=source, check=True)
print("Resolved source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip())

install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(source / "solutions/iiixr/kaggle/requirements.txt")]
subprocess.run(install + (["--no-index", "--find-links", WHEELHOUSE] if WHEELHOUSE else []), check=True)


## 3. Thực thi Full 18 câu & xuất archive + viewer offline

In [ ]:
import runpy
sys.path.insert(0, str(source))
launch = runpy.run_path(str(source / "solutions/iiixr/kaggle/notebook.py"))["launch"]
SUMMARY = launch(source, "full", globals())
